<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_2_3%D8%8C_%D8%B7%D8%B1%D9%8A%D9%82%D8%A9_Benjamini_Hochberg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 2.3: طريقة Benjamini-Hochberg

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الثاني

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 2.3».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import statsmodels.formula.api as smf
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests


def tidy(fit, terms=None):
    """جدول مختصر لنتيجة الانحدار: المعامل، والخطأ المعياري، وفترة الثقة، وقيمة p."""
    ci = fit.conf_int()
    out = pd.DataFrame({"المعامل": fit.params, "SE": fit.bse, "أدنى 95%": ci[0], "أعلى 95%": ci[1], "p": fit.pvalues})
    return out.loc[terms] if terms else out


say("الإعداد جاهز")

<div dir="rtl">

## المطلوب

طبّق طريقة Benjamini-Hochberg **يدويًا** على قيم p التالية، ثم قارن نتيجتك بالدالة `multipletests`:

`[0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.074, 0.205]`

الخطوات: رتب القيم من الأصغر إلى الأكبر، وقارن القيمة رقم k بالحد (k ÷ m) × α، ثم ارفض كل القيم حتى آخر k تحقق الشرط.

</div>

In [ ]:
pvals = np.array([0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.074, 0.205])
# اكتب حلك هنا
alpha = 0.05
m = len(pvals)

p_sorted = np.sort(pvals)
k = np.arange(1, m + 1)
limits = (k / m) * alpha

passed = p_sorted <= limits
last_k = np.where(passed)[0]

reject_manual = np.zeros(m, dtype=bool)
if len(last_k) > 0:
    reject_manual = pvals <= p_sorted[last_k[-1]]

reject_lib, _, _, _ = multipletests(
    pvals, alpha=0.05, method="fdr_bh"
)

print("النتيجة اليدوية :", reject_manual)
print("نتيجة الدالة : ", reject_lib)
print("هل متطابقتان؟ :", np.array_equal(reject_manual, reject_lib))

النتيجة اليدوية : [ True  True False False False False False False]
نتيجة الدالة :  [ True  True False False False False False False]
هل متطابقتان؟ : True


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 2.3».

</div>